In [1]:
!pip install roboflow
#/bin/python3 -m pip install ipykernel -U --user --force-reinstall


Defaulting to user installation because normal site-packages is not writeable
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.9/49.9 MB 24.5 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.2/4.2 MB 23.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 23.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 14.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.6/6.6 MB 23.0 MB/s eta 0:00:0000:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.6/8.6 MB 24.0 MB/s eta 0:00:0000:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.8/4.8 MB 23.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 15/15 [roboflow]/15 [roboflow]toolbelt]less]


In [7]:
# # Model Duckies
# !pip install roboflow

# from roboflow import Roboflow
# rf = Roboflow(api_key="ZUyvS9QdMs5yBJyz0JA3")
# project = rf.workspace("robogistics-hagpe").project("duckietown-duckies-4ew0a")
# version = project.version(4)
# dataset = version.download("yolov11")


# Model Duckies and Bots
#!pip install roboflow

from roboflow import Roboflow
#schlechter Datensatz
# rf = Roboflow(api_key="ZUyvS9QdMs5yBJyz0JA3")
# project = rf.workspace("robogistics-hagpe").project("robogistic")
# version = project.version(2)
# dataset = version.download("yolov11")
# Datensatz Duckies und Bots aber unausgegleichen 
# rf = Roboflow(api_key="ZUyvS9QdMs5yBJyz0JA3")
# project = rf.workspace("robogistics-hagpe").project("robogistic")
# version = project.version(4)
# dataset = version.download("yolov11")

rf = Roboflow(api_key="ZUyvS9QdMs5yBJyz0JA3")
project = rf.workspace("robogistics-hagpe").project("robogistic")
version = project.version(6)
dataset = version.download("yolov11")                
                

loading Roboflow workspace...
loading Roboflow project...



Extracting Dataset Version Zip to Robogistic-6 in yolov11:: 100%|██████████| 454/454 [00:00<00:00, 6898.13it/s]


In [ ]:
import os
from collections import Counter

def parse_labels(label_path):
    class_counts = Counter()
    for filename in os.listdir(label_path):
        if filename.endswith(".txt"):
            with open(os.path.join(label_path, filename), "r") as f:
                for line in f:
                    cls = line.strip().split()[0]
                    class_counts[cls] += 1
    return class_counts

# Analyse pro Split
splits = ['train', 'valid', 'test']
for split in splits:
    label_dir = os.path.join(dataset.location, split, "labels")
    counts = parse_labels(label_dir)
    total = sum(counts.values())
    print(f"\n📂 Split: {split}")
    print(f"➡️  {total} Objekte in {len(os.listdir(label_dir))} Bildern")
    print("🔢 Klassenverteilung:")
    for cls, count in counts.items():
        print(f"  - Klasse {cls}: {count}")


📂 Split: train
➡️  309 Objekte in 168 Bildern
🔢 Klassenverteilung:
  - Klasse 0: 232
  - Klasse 1: 77

📂 Split: valid
➡️  57 Objekte in 26 Bildern
🔢 Klassenverteilung:
  - Klasse 0: 45
  - Klasse 1: 12

📂 Split: test
➡️  56 Objekte in 27 Bildern
🔢 Klassenverteilung:
  - Klasse 1: 12
  - Klasse 0: 44


In [9]:
import os
import glob

def remove_class_from_labels(dataset_path, class_id_to_remove):
    for split in ["train", "valid", "test"]:
        label_dir = os.path.join(dataset_path, split, "labels")
        if not os.path.exists(label_dir):
            continue

        files = glob.glob(os.path.join(label_dir, "*.txt"))
        removed, kept = 0, 0

        for file_path in files:
            with open(file_path, 'r') as f:
                lines = f.readlines()

            # Filter: Nur Zeilen behalten, die NICHT die Klasse enthalten
            new_lines = [line for line in lines if not line.strip().startswith(str(class_id_to_remove))]

            if len(new_lines) != len(lines):
                removed += 1

            if new_lines:
                with open(file_path, 'w') as f:
                    f.writelines(new_lines)
                kept += 1
            else:
                # Falls Datei jetzt leer ist: leere Datei beibehalten oder löschen
                open(file_path, 'w').close()  # nur leeren

        print(f"[{split}] ✅ {removed} Dateien bereinigt. {kept} Dateien behalten.")

# Beispielnutzung:
remove_class_from_labels("/home/robin/Dokumente/Robogistics/Projekte_Repo/Quackademics_repo/JupyterNotebook/Robogistic-6", class_id_to_remove=1)


[train] ✅ 42 Dateien bereinigt. 168 Dateien behalten.
[valid] ✅ 7 Dateien bereinigt. 26 Dateien behalten.
[test] ✅ 8 Dateien bereinigt. 25 Dateien behalten.


In [10]:
import os
from collections import Counter

def parse_labels(label_path):
    class_counts = Counter()
    for filename in os.listdir(label_path):
        if filename.endswith(".txt"):
            with open(os.path.join(label_path, filename), "r") as f:
                for line in f:
                    if line.strip():  # ignoriert leere Zeilen
                        cls = line.strip().split()[0]
                        class_counts[cls] += 1
    return class_counts

# 🔧 DEIN PFAD HIER
dataset_root = "/home/robin/Dokumente/Robogistics/Projekte_Repo/Quackademics_repo/JupyterNotebook/Robogistic-6"

# Analyse pro Split
splits = ['train', 'valid', 'test']
for split in splits:
    label_dir = os.path.join(dataset_root, split, "labels")
    image_dir = os.path.join(dataset_root, split, "images")

    if not os.path.exists(label_dir):
        print(f"❌ Split '{split}' nicht gefunden – übersprungen.")
        continue

    counts = parse_labels(label_dir)
    total_objects = sum(counts.values())
    total_images = len([f for f in os.listdir(image_dir) if f.lower().endswith(('.jpg', '.png'))])

    print(f"\n📂 Split: {split}")
    print(f"➡️  {total_objects} Objekte in {total_images} Bildern")
    print("🔢 Klassenverteilung:")
    for cls, count in sorted(counts.items(), key=lambda x: int(x[0])):
        print(f"  - Klasse {cls}: {count}")



📂 Split: train
➡️  232 Objekte in 168 Bildern
🔢 Klassenverteilung:
  - Klasse 0: 232

📂 Split: valid
➡️  45 Objekte in 26 Bildern
🔢 Klassenverteilung:
  - Klasse 0: 45

📂 Split: test
➡️  44 Objekte in 27 Bildern
🔢 Klassenverteilung:
  - Klasse 0: 44


In [12]:
%pip install ultralytics

Defaulting to user installation because normal site-packages is not writeable
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 12.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 67.0/67.0 MB 19.8 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.3/12.3 MB 21.2 MB/s eta 0:00:0000:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 37.7/37.7 MB 9.7 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 821.2/821.2 MB ? eta 0:00:00 0:00:0100:02
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 393.1/393.1 MB 3.3 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.9/8.9 MB 18.8 MB/s eta 0:00:0000:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.7/23.7 MB 24.2 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 897.7/897.7 kB 15.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 571.0/571.0 MB ? eta 0:00:00 0:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 200.2/200.2 MB 7.6 M

In [ ]:

from ultralytics import YOLO

model = YOLO('yolo11n.pt') # kleinste Architektur

model.train(
    data='/home/robin/Dokumente/Robogistics/Projekte_Repo/Quackademics_repo/JupyterNotebook/data.yaml',
    epochs=50,
    imgsz=640,
    batch=8,
    patience=20,  # Frühes Stoppen, falls kein Fortschritt
    lr0=0.001,    # Anfangs-Lernrate
    lrf=0.01,     # finaler Faktor der Lernrate
    warmup_epochs=5,  # hilft bei stabilerem Start
    weight_decay=0.0005,  # wichtig gegen Overfitting
    mosaic=0.2,   # begrenzt Augmentation (nicht zu stark!)
    hsv_h=0.015, hsv_s=0.7, hsv_v=0.4,  # Farben leicht variieren
    degrees=5, translate=0.05, scale=0.2, shear=2.0,  # moderate Geometrie
    flipud=0.0, fliplr=0.5,
    dropout=0.1,  # aktivieren, falls du es unterstützt (custom head)
)

Ultralytics 8.3.167 🚀 Python-3.10.12 torch-2.7.1+cu126 CPU (Intel Core(TM) i5-1035G4 1.10GHz)
engine/trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/home/robin/Dokumente/Robogistics/Projekte_Repo/Quackademics_repo/JupyterNotebook/data.yaml, degrees=5, deterministic=True, device=cpu, dfl=1.5, dnn=False, dropout=0.1, dynamic=False, embed=None, epochs=100, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.001, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo11n.pt, momentum=0.937, mosaic=0.2, multi_scale=False, name=train2, nbs=64, nms=False, opset=None, optimize=False, optimizer=auto, overlap_ma

train: Scanning /home/robin/Dokumente/Robogistics/Projekte_Repo/Quackademics_repo/JupyterNotebook/train/labels.cache... 1628 images, 561 backgrounds, 0 corrupt: 100%|██████████| 1628/1628 [00:00<?, ?it/s]

val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 27.2±21.6 MB/s, size: 41.5 KB)



/home/robin/.local/lib/python3.10/site-packages/torch/utils/data/dataloader.py:665: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)
val: Scanning /home/robin/Dokumente/Robogistics/Projekte_Repo/Quackademics_repo/JupyterNotebook/valid/labels.cache... 366 images, 84 backgrounds, 0 corrupt: 100%|██████████| 366/366 [00:00<?, ?it/s]
/home/robin/.local/lib/python3.10/site-packages/torch/utils/data/dataloader.py:665: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


Plotting labels to runs/detect/train2/labels.jpg... 
optimizer: 'optimizer=auto' found, ignoring 'lr0=0.001' and 'momentum=0.937' and determining best 'optimizer', 'lr0' and 'momentum' automatically... 
optimizer: AdamW(lr=0.002, momentum=0.9) with parameter groups 81 weight(decay=0.0), 88 weight(decay=0.0005), 87 bias(decay=0.0)
Image sizes 640 train, 640 val
Using 0 dataloader workers
Logging results to runs/detect/train2
Starting training for 100 epochs...

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size


      1/100         0G      1.332      2.506     0.9013         33        640:  42%|████▏     | 85/204 [05:32<07:45,  3.91s/it]


In [ ]:
# Voraussetzung: Installation der Ultralytics-Bibliothek (YOLOv11)
# Installieren (falls noch nicht vorhanden):
!pip install ultralytics

from ultralytics import YOLO

# === Schritt 1: Modell laden ===
model = YOLO('yolo11n.pt')  # Verwende YOLOv11 Nano (leichtes Modell für Edge Devices)

# === Schritt 2: Training starten ===
model.train(
    data='data.yaml',     # Pfad zu deiner data.yaml-Datei
    epochs=50,            # Anzahl der Epochen
    imgsz=416,            # Eingabebildgröße (416x416 schneller als 640)
    batch=16,             # Batch-Größe
    name='duckie11_train', # Name des Runs / Ausgabeordners
    device = 0
)

# === Schritt 3: Exportiere bestes Modell als ONNX ===
model.export(
    format='onnx',  # Export-Format
    dynamic=True,   # Erlaubt variable Eingabegrößen
    simplify=True   # Optimiert das ONNX-Modell
)

print("✅ Training abgeschlossen und Modell als ONNX exportiert.")


In [ ]:
import torch

if torch.cuda.is_available():
    num_devices = torch.cuda.device_count()
    for i in range(num_devices):
        print(f"Device {i}: {torch.cuda.get_device_name(i)}")
else:
    print("❌ Keine CUDA-fähige GPU gefunden.")